**Notebook Status:** Work in Progress

# Visualizing Holographic Reduced Representation Binding

Here we compare the effect of iterated circular convolution on random normalized (unit) HRR vectors and random unitary HRR vectors. For each vector $x$ at each step $t$, we visualize the result of the map $x\mapsto x^t=x\circledast x\circledast\ldots\circledast x$.

The difference is striking. In general, points on the sphere expand outwards, converging to subspaces of $\mathbb{R}^n$, whereas unitary vectors stay fixed in a (disjoint) subset. In particular we can see how circular convolution does not act on the spherical vectors in a uniform manner. 

In [ ]:
# Install required packages (if required)
import sys
!{sys.executable} -m pip install numpy plotly

In [ ]:
import numpy as np
import plotly.graph_objects as go

In [ ]:
def visualize_iterated_binding(n_sphere, n_unitary, steps=25, N=3)->go.Figure:
    # Generate random unitary points
    u_points = np.fft.fft(np.random.normal(0,1,(n_unitary, N)), axis=-1)
    u_points = u_points/np.abs(u_points)

    # Generate random unitary points
    s_points = np.random.normal(0,1,(n_sphere, N))
    s_points = np.fft.fft(np.einsum('ij,i->ij', s_points, 1/np.linalg.norm(s_points, axis=-1)), axis=-1)

    # Empty ndarrays to store data
    u_data = np.empty((steps, n_unitary, N), dtype=complex)
    p_data = np.empty((steps, n_sphere, N), dtype=complex)

    # Transform points
    for step in range(steps):
        u_data[step, :, :] = u_points**(step+1)
        p_data[step, :, :] = s_points**(step+1)

    # Convert data back to real 3D vectors
    p_data = np.fft.ifft(p_data, axis=-1).real
    u_data = np.fft.ifft(u_data, axis=-1).real

    # Visualization
    frames = [go.Frame(
        data = [
            go.Scatter3d(
                x = p_data[i,:,0], 
                y = p_data[i,:,1], 
                z = p_data[i,:,2], 
                mode = 'markers', 
                marker = dict(size=3), 
                name = 'random sphere'),
            go.Scatter3d(
                x = u_data[i,:,0], 
                y = u_data[i,:,1], 
                z = u_data[i,:,2], 
                mode = 'markers', 
                marker = dict(size=3), 
                name = 'random unitary'),
        ], name = str(i)) 
        for i in range(steps)]
    fig = go.Figure(frames = frames)
    fig.add_traces([
        go.Scatter3d(
            x = p_data[0,:,0], 
            y = p_data[0,:,1], 
            z = p_data[0,:,2], 
            mode = 'markers', 
            marker = dict(size=3), 
            name = 'random sphere'
        ),
        go.Scatter3d(
            x = u_data[0,:,0], 
            y = u_data[0,:,1], 
            z = u_data[0,:,2], 
            mode = 'markers', 
            marker = dict(size=3), 
            name = 'random unitary'
        ),
    ])
    sliders =[{
        "steps": [
            {
            "args": [
                [f.name], {
                    "frame": {"duration": 0},
                    "mode": "immediate",
                    "fromcurrent": True,
                }],
            "label": str(k),
            "method": "animate",
            }
            for k, f in enumerate(fig.frames)
        ],
    }]
    fig.update_layout(
        sliders=sliders,
        scene = dict(
            xaxis = dict(range=[-3, 3]),
            yaxis = dict(range=[-3, 3]),
            zaxis = dict(range=[-3, 3]),
            aspectratio = dict(x=1, y=1, z=1),
        )
    )
    return fig

### 3 Dimensional Unit Vectors

In [ ]:
visualize_iterated_binding(500, 200).show()

### 4 Dimensional Unit Vectors
*Note: This is a 3D slice of 4D space*

In [ ]:
visualize_iterated_binding(500, 200, N=4).show()

### 5 Dimensional Unit Vectors
*Note: This is a 3D slice of 5D space*

In [ ]:
visualize_iterated_binding(500, 500, N=5).show()